# Tahap 1 — Prapemrosesan: Normalisasi Rasio B/R (`05b_densenet121_Freeze_weight`)

Notebook ini **membangun dan memeriksa Tahap 1** pipeline eksperimen DenseNet-121
*freeze weight*: `X_norm = BR_Ratio_Normalization(Image_RGB_512x512)`.
Lihat rancangan di `docs/pelatihan/br_ratio_normalization/BR_RATIO_NORMALIZATION.md` (§2).

**Tujuan sel-sel di bawah:**
1. Mendefinisikan transform B/R yang **portabel** (bisa dipindah ke `src/datasets/` pada tahap lanjut).
2. Mengambil **5 sampel** (satu per kelas) lalu menampilkan `asli | λ_N | X_norm`.
3. **Menyimpan** hasil (`tahap1_samples.pt`, `tahap1_config.json`, PNG preview) agar bisa
   dipakai tahap berikutnya (model beku + pelatihan).

> **PENTING — jangan auto-run.** Notebook ini sengaja **belum dijalankan** (tanpa output).
> Jalankan sel dari atas ke bawah saat Anda siap. Tidak ada kode `src/` yang diubah;
> transform B/R didefinisikan di sini sebagai draft yang dapat dipromosikan nanti.

> **Keputusan tertunda (dari draft §2.1/§9):** definisi `Reconstruct_Image` belum final.
> Notebook memakai variabel `RECONSTRUCT_MODE` (default `"replicate"`) agar mudah diganti
> tanpa mengubah logika lain.

## 1. Setup & path repo

In [1]:
import os, sys, json
from pathlib import Path

# Pastikan berjalan dari akar repo (folder yang memuat src/).
REPO = Path.cwd()
for _ in range(6):
    if (REPO / 'src').exists():
        break
    REPO = REPO.parent
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms

print('repo :', REPO)
print('torch:', torch.__version__)

repo : d:\n8n-logsiswaparalayang\Jurnal\CitraAwan
torch: 2.1.1+cu121


## 2. Konfigurasi Tahap 1

Ubah variabel bila perlu. `RECONSTRUCT_MODE` adalah keputusan tertunda dari draft:
- `"replicate"`  — `λ_N` di-remap ke [0,1] lalu disalin ke 3 kanal (kompatibel conv0 ImageNet 3-kanal).
- `"channel_swap"` — `X_norm = [R, G, λ_N(0..1)]` (dua kanal asli dipertahankan, biru diganti λ_N).

Keduanya menjaga **3 kanal** sehingga bobot ImageNet (backbone beku) tetap dipakai apa adanya.

In [ ]:
# --- sumber data (train seimbang combined_balanced; sama dengan exp_005) ---
TRAIN_DIR = REPO / 'data' / 'combined_balanced' / 'train'
TRAIN_MANIFEST = REPO / 'reports' / 'tables' / 'combined_balanced_train_manifest.csv'
CLASS_NAMES = ['Cu', 'Ac', 'Ci', 'Cl', 'Sc', 'Cb', 'Mi']  # urutan label 0..6 (configs/base.yaml)

# --- parameter Tahap 1 ---
IMAGE_SIZE = 512    
EPS = 1e-5                          # pencegah bagi-nol pada (b + r + eps)
RECONSTRUCT_MODE = 'replicate'      # 'replicate' | 'channel_swap'  (lihat draft §2.1/§9)
REMAP_TO_01 = True                  # remap λ_N dari (-1,1) ke [0,1] agar rentang citra valid
N_SAMPLES = 5                       # tampilkan 5 sampel

# --- output ARTEFAK -> experiments/ (bukan docs; docs hanya untuk dokumen/analisis) ---
OUT_DIR = REPO / 'experiments' / 'exp_005b_densenet121_freeze' / 'tahap1_prapemrosesan'
OUT_DIR.mkdir(parents=True, exist_ok=True)
PREVIEW_PNG = OUT_DIR / 'tahap1_samples_preview.png'
SAMPLES_PT = OUT_DIR / 'tahap1_samples.pt'
CONFIG_JSON = OUT_DIR / 'tahap1_config.json'
print('TRAIN_DIR exists :', TRAIN_DIR.exists())
print('MANIFEST exists  :', TRAIN_MANIFEST.exists())
print('OUT_DIR          :', OUT_DIR)

TRAIN_DIR exists : True
MANIFEST exists  : True
OUT_DIR          : d:\n8n-logsiswaparalayang\Jurnal\CitraAwan\experiments\exp_005b_densenet121_freeze\tahap1_prapemrosesan


## 3. Definisi transform Normalisasi Rasio B/R (portabel)

Implementasi `ALGORITHM 1` dari draft. Fungsi ini **murni tensor** sehingga dapat
dipindah ke `src/datasets/cloud_dataset.py` (mis. sebagai `normalize: br_ratio`) pada tahap lanjut.

In [3]:
def br_ratio_normalize(x: torch.Tensor, eps: float = EPS,
                       mode: str = RECONSTRUCT_MODE,
                       remap: bool = REMAP_TO_01):
    """Normalisasi Rasio B/R (invarian cahaya) pada tensor RGB CxHxW di [0,1].

    Mengembalikan (X_norm [3xHxW], lambda_N [1xHxW]).
    Urutan kanal torchvision: 0=R, 1=G, 2=B.
    """
    assert x.ndim == 3 and x.shape[0] == 3, 'harap tensor RGB 3xHxW'
    r = x[0]
    b = x[2]
    lam = (b - r) / (b + r + eps)          # rentang teoretis (-1, 1)
    lam = lam.unsqueeze(0)                  # 1xHxW
    lam01 = (lam + 1.0) / 2.0 if remap else lam
    if mode == 'replicate':
        x_norm = lam01.repeat(3, 1, 1)
    elif mode == 'channel_swap':
        x_norm = torch.stack([x[0], x[1], lam01.squeeze(0)], dim=0)
    else:
        raise ValueError(f'RECONSTRUCT_MODE tidak dikenal: {mode!r}')
    return x_norm.clamp(0.0, 1.0), lam

# Transform pemuatan: PIL -> resize -> ToTensor([0,1]) (B/R diterapkan terpisah di bawah).
load_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])

def load_rgb01(path):
    with Image.open(path) as im:
        return load_tf(im.convert('RGB'))

## 4. Ambil 5 sampel (satu per kelas)

Memakai pembaca manifes proyek (`src.datasets.manifest`) agar resolusi path konsisten
dengan pipeline pelatihan.

In [4]:
from src.datasets.manifest import load_samples

samples = load_samples(TRAIN_MANIFEST, TRAIN_DIR)

# Pilih 1 sampel pertama untuk tiap kelas, hingga N_SAMPLES kelas berbeda.
picked, seen = [], set()
for s in samples:
    if s.label not in seen:
        picked.append(s)
        seen.add(s.label)
    if len(picked) >= N_SAMPLES:
        break

print(f'total sampel manifes: {len(samples)}')
for s in picked:
    print(f'  label {s.label} ({CLASS_NAMES[s.label]}):', s.path.name)

total sampel manifes: 21000
  label 0 (Cu): 1_cumulus_000002.jpg
  label 1 (Ac): 2_altocumulus_000001.jpg
  label 2 (Ci): 3_cirrus_000003.jpg
  label 3 (Cl): 4_clearsky_000001.jpg
  label 4 (Sc): 5_stratocumulus_000001.jpg


## 5. Terapkan Tahap 1 & tampilkan 5 sampel

Baris = sampel; kolom = `asli (RGB)` | `λ_N (peta B/R)` | `X_norm (masukan model)`.

In [5]:
rows = len(picked)
fig, axes = plt.subplots(rows, 3, figsize=(9, 3 * rows))
if rows == 1:
    axes = axes[None, :]

X_list, lam_list = [], []
for i, s in enumerate(picked):
    rgb = load_rgb01(s.path)                     # 3xHxW [0,1]
    x_norm, lam = br_ratio_normalize(rgb)        # 3xHxW, 1xHxW
    X_list.append(x_norm)
    lam_list.append(lam)

    axes[i, 0].imshow(rgb.permute(1, 2, 0).numpy())
    axes[i, 0].set_ylabel(f'{s.label} · {CLASS_NAMES[s.label]}', fontsize=10)
    axes[i, 0].set_title('asli (RGB)' if i == 0 else '')

    axes[i, 1].imshow(lam.squeeze(0).numpy(), cmap='coolwarm', vmin=-1, vmax=1)
    axes[i, 1].set_title('λ_N = (B-R)/(B+R+ε)' if i == 0 else '')

    axes[i, 2].imshow(x_norm.permute(1, 2, 0).numpy())
    axes[i, 2].set_title(f'X_norm ({RECONSTRUCT_MODE})' if i == 0 else '')

    for j in range(3):
        axes[i, j].set_xticks([]); axes[i, j].set_yticks([])

fig.suptitle('Tahap 1 — Normalisasi Rasio B/R (5 sampel)', y=1.0)
fig.tight_layout()
fig.savefig(PREVIEW_PNG, dpi=120, bbox_inches='tight')
print('tersimpan:', PREVIEW_PNG)
plt.show()

tersimpan: d:\n8n-logsiswaparalayang\Jurnal\CitraAwan\experiments\exp_005b_densenet121_freeze\tahap1_prapemrosesan\tahap1_samples_preview.png


<Figure size 900x1500 with 15 Axes>

## 6. Simpan hasil untuk tahap selanjutnya

Menyimpan tensor `X_norm` 5 sampel + peta `λ_N` + metadata, serta config Tahap 1,
ke `experiments/exp_005b_densenet121_freeze/tahap1_prapemrosesan/` (artefak dicatat di `experiments/`).
Berkas ini bisa dimuat ulang oleh tahap pelatihan **tanpa menghitung ulang**:

```python
blob = torch.load('experiments/exp_005b_densenet121_freeze/tahap1_prapemrosesan/tahap1_samples.pt')
X = blob['X_norm']   # [5, 3, 512, 512], siap jadi masukan model beku
```

In [6]:
X = torch.stack(X_list, dim=0)                 # [N, 3, H, W]
lam = torch.stack(lam_list, dim=0)             # [N, 1, H, W]
labels = torch.tensor([s.label for s in picked])

torch.save({
    'X_norm': X,
    'lambda_N': lam,
    'labels': labels,
    'class_names': CLASS_NAMES,
    'paths': [str(s.path) for s in picked],
    'reconstruct_mode': RECONSTRUCT_MODE,
    'eps': EPS,
    'remap_to_01': REMAP_TO_01,
}, SAMPLES_PT)

cfg = {
    'stage': 'tahap1_br_ratio_normalization',
    'formula': 'lambda_N = (B - R) / (B + R + eps)',
    'eps': EPS,
    'reconstruct_mode': RECONSTRUCT_MODE,
    'remap_to_01': REMAP_TO_01,
    'image_size': IMAGE_SIZE,
    'channel_order': 'RGB (0=R,1=G,2=B)',
    'note': 'Draft Tahap 1; belum dipromosikan ke src/datasets. Lihat BR_RATIO_NORMALIZATION.md.',
}
with open(CONFIG_JSON, 'w', encoding='utf-8') as f:
    json.dump(cfg, f, ensure_ascii=False, indent=2)

print('tersimpan:', SAMPLES_PT)
print('tersimpan:', CONFIG_JSON)
print('X_norm shape:', tuple(X.shape), '| lambda_N shape:', tuple(lam.shape))
print('rentang X_norm: [%.3f, %.3f]' % (X.min().item(), X.max().item()))
print('rentang lambda_N: [%.3f, %.3f]' % (lam.min().item(), lam.max().item()))

tersimpan: d:\n8n-logsiswaparalayang\Jurnal\CitraAwan\experiments\exp_005b_densenet121_freeze\tahap1_prapemrosesan\tahap1_samples.pt
tersimpan: d:\n8n-logsiswaparalayang\Jurnal\CitraAwan\experiments\exp_005b_densenet121_freeze\tahap1_prapemrosesan\tahap1_config.json
X_norm shape: (5, 3, 512, 512) | lambda_N shape: (5, 1, 512, 512)
rentang X_norm: [0.476, 1.000]
rentang lambda_N: [-0.048, 1.000]


## 7. Catatan & keputusan tertunda

- Analisis visual/statistik 5 sampel ditulis di
  `docs/pelatihan/densenet121_freeze/tahap1_prapemrosesan/DRAFT_ANALISIS_TAHAP1.md`.
- Artefak (`.pt`/`.png`/`.json`) tercatat di `experiments/exp_005b_densenet121_freeze/tahap1_prapemrosesan/`.
- **Definisi `Reconstruct_Image`** (`RECONSTRUCT_MODE`) masih perlu disepakati (draft §2.1/§9).
- Backbone ImageNet **beku** mengharapkan statistik RGB; masukan B/R mengubah distribusi —
  amati apakah fitur beku tetap informatif atau tahap ini menuntut `finetune: full`.
- Promosi ke pipeline pelatihan: pindahkan `br_ratio_normalize` ke `src/datasets/cloud_dataset.py`
  dan tambah `preprocess.normalize: br_ratio` pada config eksperimen (BELUM dilakukan).